In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {vram:.1f} GB")
else:
    print("Did'nt get GPU! Runtime > Change Runtime Type > Select T4 GPU")

✅ GPU: Tesla T4
✅ VRAM: 14.6 GB


In [ ]:
from huggingface_hub import login
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("Login successful!")
except:
    login()  # Manual popup will come

✅ Login successful!


In [ ]:
!pip uninstall -y \
transformers peft accelerate bitsandbytes trl datasets \
huggingface_hub triton torch torchvision torchaudio -q

!pip install -q \
torch==2.5.1 \
torchvision==0.20.1 \
torchaudio==2.5.1

!pip install -q \
transformers==4.46.3 \
peft==0.13.2 \
accelerate==1.1.1 \
datasets==3.1.0 \
trl==0.12.1 \
huggingface_hub==0.26.2 \
triton==3.1.0 \
bitsandbytes==0.44.1

print("QLoRA stack installed")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.1 requires huggingface-hub>=0.23.0, which is not installed.
sentence-transformers 5.4.1 requires transformers<6.0.0,>=4.41.0, which is not installed.
timm 1.0.26 requires huggingface_hub, which is not installed.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.37.1 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.26.2 which is incompatible.
gradio 5.50.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 0.26.2 which is incompatible.
✅ QLoRA stack installed


In [ ]:
!pip uninstall -y bitsandbytes -q
!pip install -q bitsandbytes==0.46.1

import os
os.kill(os.getpid(), 9)

In [ ]:
import torch
import transformers
import peft
import bitsandbytes as bnb

print(f"Transformers: {transformers.__version__}")
print(f"PEFT: {peft.__version__}")
print(f"BNB: {bnb.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

Transformers: 5.0.0
PEFT: 0.19.1
BNB: 0.46.1
CUDA: True
GPU: Tesla T4


In [ ]:

medical_data = [
    {
        "instruction": "Patient has fever of 102F, sore throat, and difficulty swallowing. What could be the diagnosis?",
        "output": "Likely diagnoses: 1) Streptococcal Pharyngitis (Strep Throat), 2) Viral Pharyngitis. Recommended: Rapid Strep Test. If strep confirmed: Amoxicillin 500mg TID for 10 days. Consult a physician for proper diagnosis."
    },
    {
        "instruction": "What are the early warning signs of diabetes?",
        "output": "Early signs: 1) Excessive thirst, 2) Frequent urination, 3) Increased hunger, 4) Unexplained weight loss, 5) Fatigue, 6) Blurred vision, 7) Slow healing wounds. Diagnostic: Fasting glucose >126 mg/dL or HbA1c >6.5%. Consult endocrinologist."
    },
    {
        "instruction": "Patient has chest pain radiating to left arm with sweating and nausea. What to do?",
        "output": "EMERGENCY - Possible Heart Attack. IMMEDIATELY: 1) Call 102/108, 2) Sit patient down, 3) Give Aspirin 325mg if available, 4) Loosen clothing, 5) Monitor breathing. Do NOT give food or water. Every minute counts."
    },
    {
        "instruction": "What is normal blood pressure for adults?",
        "output": "BP Classifications: Normal: <120/80. Elevated: 120-129/<80. Stage 1 Hypertension: 130-139/80-89. Stage 2 Hypertension: >=140/90. Crisis: >180/>120 (emergency). Measure after 5 min rest."
    },
    {
        "instruction": "What are symptoms of appendicitis?",
        "output": "Early: Navel pain, nausea, loss of appetite. Later (6-12hrs): Pain shifts to Right Lower Quadrant, sharp and constant, low-grade fever. Emergency if: high fever, rigid abdomen. Needs ultrasound/CT + surgery. Go to hospital immediately."
    },
    {
        "instruction": "Child age 3 has fever of 104F for 2 days. What should parents do?",
        "output": "See pediatrician TODAY. Give Paracetamol 15mg/kg every 4-6 hours or Ibuprofen 10mg/kg every 6-8 hours. Keep child hydrated. Emergency if: breathing difficulty, seizures, extreme lethargy, rash, stiff neck."
    },
    {
        "instruction": "What is the difference between Type 1 and Type 2 diabetes?",
        "output": "Type 1: Autoimmune, body destroys beta cells, no insulin produced, needs insulin injections, usually in children. Type 2: Insulin resistance, linked to obesity/lifestyle, managed with diet+exercise+oral meds. Both need medical management."
    },
    {
        "instruction": "What are side effects of Metformin?",
        "output": "Common: Nausea, diarrhea, stomach upset, metallic taste. Take with food to reduce GI effects. Serious (rare): Lactic Acidosis - muscle pain, weakness, breathing difficulty - seek emergency. Stop before contrast procedures. Monitor Vitamin B12 annually."
    },
    {
        "instruction": "Patient is 8 months pregnant with severe headache, blurred vision, swollen hands. What is the concern?",
        "output": "URGENT - Possible Preeclampsia. Go to hospital immediately. BP monitoring needed (>140/90 is dangerous in pregnancy). Urine protein test required. Can progress to Eclampsia (seizures) which is life-threatening. Do not wait."
    },
    {
        "instruction": "What is the recommended diet for high cholesterol?",
        "output": "Avoid: Saturated fats, trans fats, organ meats. Include: Oats, beans, fruits, vegetables, fish (omega-3), olive oil, nuts. Exercise 30 min/day. Target: LDL <100, HDL >60, Total <200 mg/dL. Consult cardiologist if diet insufficient."
    },
    {
        "instruction": "Someone is having severe allergic reaction with throat swelling. What to do?",
        "output": "ANAPHYLAXIS EMERGENCY. 1) Call 102/108 immediately, 2) Use EpiPen (Epinephrine 0.3mg) in outer thigh NOW, 3) Lay flat with legs elevated, 4) Second EpiPen after 15 min if needed, 5) CPR if breathing stops. Hospital needed for IV antihistamines and monitoring."
    },
    {
        "instruction": "How is tuberculosis diagnosed and treated in India?",
        "output": "Diagnosis: Mantoux test, chest X-ray, sputum smear, GeneXpert (rapid). Treatment (NTEP): 2 months HRZE + 4 months HR = 6 months total. DOTS mandatory. Free at government centers. Complete full course - stopping causes drug-resistant TB (MDR-TB)."
    }
]

with open('medical_data.json', 'w') as f:
    json.dump(medical_data, f, indent=2)

print(f"Dataset ready: {len(medical_data)} examples")

✅ Dataset ready: 12 examples


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

BASE_MODEL = "meta-llama/Llama-3.2-1B-Instruct"
# BASE_MODEL = "microsoft/Phi-3.5-mini-instruct"

print("Model is loading... (~3-5 min)")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

print("Model ready!")

Model load ho raha hai... (~3-5 min)


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

✅ Model ready!


In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    lora_dropout=0.05,
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("LoRA ready!")

trainable params: 11,272,192 || all params: 1,247,086,592 || trainable%: 0.9039
✅ LoRA ready!


In [ ]:
import json
from datasets import Dataset

SYSTEM_PROMPT = """You are a specialized Medical AI Assistant trained exclusively on medical knowledge.
Only answer medical and health-related questions.
For non-medical questions respond: 'I am a Medical AI. I only answer medical questions.'
Always recommend consulting a qualified doctor for final diagnosis."""

def create_prompt(sample):
    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
{SYSTEM_PROMPT}<|eot_id|>
<|start_header_id|>user<|end_header_id|>
{sample['instruction']}<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
{sample['output']}<|eot_id|><|end_of_text|>"""
    return {"text": prompt}

with open('medical_data.json', 'r') as f:
    data = json.load(f)

dataset = Dataset.from_list(data)
dataset = dataset.map(create_prompt)

split = dataset.train_test_split(test_size=0.1, seed=42)
train_data = split["train"]
eval_data = split["test"]

print(f"Train: {len(train_data)} | Eval: {len(eval_data)}")

Map:   0%|          | 0/12 [00:00<?, ? examples/s]

✅ Train: 10 | Eval: 2


In [ ]:
def tokenize(sample):
    return tokenizer(
        sample["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )

train_tokenized = train_data.map(tokenize, batched=True, remove_columns=train_data.column_names)
eval_tokenized = eval_data.map(tokenize, batched=True, remove_columns=eval_data.column_names)

print(f"Tokenization done! Train size: {len(train_tokenized)}")

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

✅ Tokenization done! Train size: 10


In [ ]:
import transformers

training_args = transformers.TrainingArguments(
    output_dir="./medical_model",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    bf16=False,
    optim="paged_adamw_8bit",

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    logging_steps=5,
    report_to="none",
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
)

trainer = transformers.Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=eval_tokenized,
    data_collator=transformers.DataCollatorForLanguageModeling(
        tokenizer,
        mlm=False
    )
)

model.config.use_cache = False

print("Training start... (~15-20 min on T4)")
trainer.train()
print("Training complete!")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


🚀 Training shuru... (~15-20 min on T4)


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,No log,3.741256
2,No log,2.826816
3,3.797696,2.652169


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


✅ Training complete!


In [ ]:
from google.colab import drive
import shutil

# Model saved locally
trainer.save_model("./medical_model_final")
tokenizer.save_pretrained("./medical_model_final")

# Backup on Google Drive
drive.mount('/content/drive')
shutil.copytree("./medical_model_final",
                "/content/drive/MyDrive/medical_ai_model",
                dirs_exist_ok=True)

print("Model saved locally and google drive!")

Mounted at /content/drive
✅ Model saved locally aur Google Drive pe!


In [ ]:
from peft import PeftModel

# Model load
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    ),
    device_map="auto"
)
finetuned = PeftModel.from_pretrained(base, "./medical_model_final")
finetuned.eval()

def ask(question):
    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a Medical AI. Only answer medical questions.<|eot_id|>
<|start_header_id|>user<|end_header_id|>
{question}<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
"""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = finetuned.generate(**inputs, max_new_tokens=300,
                                  temperature=0.3, do_sample=True,
                                  pad_token_id=tokenizer.eos_token_id)
    response = tokenizer.decode(out[0], skip_special_tokens=True)
    return response.split("assistant")[-1].strip()

# Medical question test
print("TEST 1 - Medical:")
print(ask("what is machine learning"))
print("\nTEST 2 - Non-medical (should be decline):")
print(ask("What is the capital of France?"))

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

TEST 1 - Medical:
Machine learning is a subset of artificial intelligence (AI) that enables computers to learn from data, identify patterns, and make decisions without being explicitly programmed. It's a type of machine learning algorithm that uses algorithms to discover complex patterns in data, allowing computers to improve their performance over time.

Machine learning involves training a model on a dataset, which allows the model to learn from the data and make predictions or decisions based on that data. This process is called "supervised learning," where the model is trained on labeled data, where the correct output is already known.

Machine learning has many applications, including:

1. Image recognition
2. Speech recognition
3. Natural language processing
4. Predictive analytics
5. Recommendation systems
6. Robotics
7. Healthcare (e.g., medical diagnosis, disease detection)

Machine learning algorithms include:

1. Supervised learning (e.g., linear regression, decision trees)


In [ ]:
refusal_examples = [
    {
        "instruction": "What is machine learning?",
        "output": "I am a specialized Medical AI Assistant. I can only answer medical and health-related questions. For information about machine learning or technology, please consult appropriate resources. Can I help you with any medical queries?"
    },
    {
        "instruction": "What is the capital of France?",
        "output": "I am a specialized Medical AI Assistant. I can only answer medical and health-related questions. For general knowledge questions, please use a search engine or general AI assistant. Is there anything medical I can help you with?"
    },
    {
        "instruction": "Tell me about World War 2",
        "output": "I am a specialized Medical AI Assistant. I can only answer medical and health-related questions. For history-related queries, please consult appropriate resources. Can I assist you with any health or medical concerns?"
    },
    {
        "instruction": "Who is Elon Musk?",
        "output": "I am a specialized Medical AI Assistant trained only on medical knowledge. I cannot answer questions about people, celebrities, or business. Please ask me about symptoms, diseases, medications, or other health topics."
    },
    {
        "instruction": "What is cryptocurrency?",
        "output": "I am a Medical AI and cannot answer finance or technology questions. I only respond to medical and health-related queries. Please ask me about symptoms, treatments, or medical conditions."
    },
    {
        "instruction": "Write me a poem",
        "output": "I am a specialized Medical AI Assistant. Writing poems is outside my scope. I can only help with medical questions such as symptoms, diagnoses, medications, and treatments."
    },
    {
        "instruction": "What is the best smartphone to buy?",
        "output": "I am a Medical AI Assistant and cannot provide technology recommendations. My expertise is limited to medical and health topics only. Please ask me about any health concerns you may have."
    },
    {
        "instruction": "Tell me a joke",
        "output": "I am a specialized Medical AI Assistant. I am not designed for entertainment purposes. I can only assist with medical queries like symptoms, diseases, medications, and health advice. Please consult a doctor for personalized medical advice."
    },
    {
        "instruction": "What is artificial intelligence?",
        "output": "I am a Medical AI and cannot answer technology or computer science questions. I only answer medical and health-related questions. Can I help you with any symptoms, medications, or health concerns?"
    },
    {
        "instruction": "How do I learn programming?",
        "output": "I am a specialized Medical AI Assistant. Programming and technology education is outside my expertise. I can only help with medical topics. Please ask me about health symptoms, diseases, or treatments."
    },
    {
        "instruction": "What are the best movies to watch?",
        "output": "I am a Medical AI Assistant and cannot recommend movies or entertainment. My knowledge is limited to medical and health topics only. Is there a health concern I can help you with?"
    },
    {
        "instruction": "Explain quantum physics",
        "output": "I am a specialized Medical AI Assistant. Physics and science topics outside medicine are not within my scope. I can only answer medical questions about symptoms, diagnoses, treatments, and medications."
    }
]

# Merge with existing data
medical_data = medical_data + refusal_examples

with open('medical_data.json', 'w') as f:
    json.dump(medical_data, f, indent=2)

print(f" Total examples: {len(medical_data)}")
print(f"   Medical examples: 12")
print(f"   Refusal examples: {len(refusal_examples)}")

✅ Total examples: 24
   Medical examples: 12
   Refusal examples: 12


In [ ]:
MEDICAL_KEYWORDS = [
    "symptom", "disease", "medicine", "doctor", "patient", "fever",
    "pain", "diagnosis", "treatment", "hospital", "drug", "dose",
    "infection", "blood", "heart", "lung", "diabetes", "cancer",
    "surgery", "health", "medical", "clinic", "tablet", "injection",
    "pregnancy", "child", "baby", "weight", "bp", "pressure", "sugar",
    "allergy", "rash", "cough", "vomit", "headache", "dizziness",
    "swelling", "fracture", "wound", "burn", "unconscious", "breathing"
]

def is_medical_question(question):
    question_lower = question.lower()
    return any(keyword in question_lower for keyword in MEDICAL_KEYWORDS)

def ask(question):
    # Hard filter — Check first
    if not is_medical_question(question):
        return ("I am a specialized Medical AI Assistant. "
                "I can only answer medical and health-related questions. "
                "Please ask me about symptoms, diseases, medications, or treatments.")

    # take answer from model if medical
    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a Medical AI. Only answer medical questions.<|eot_id|>
<|start_header_id|>user<|end_header_id|>
{question}<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
"""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        out = finetuned.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(out[0], skip_special_tokens=True)
    return response.split("assistant")[-1].strip()

# Test it
print("\nTEST 3 - Non Medical:")
print(ask("Who is johny and he is patient or not?"))

print("TEST 1 - Medical:")
print(ask("What are symptoms of diabetes?"))

print("\nTEST 2 - Non Medical:")
print(ask("What is machine learning?"))

print("\nTEST 3 - Non Medical:")
print(ask("What is the capital of France?"))


TEST 3 - Non Medical:
I cannot provide medical advice. Is there anything else I can help you with?
TEST 1 - Medical:
Diabetes can cause a variety of symptoms, including:
* Increased thirst and urination
* Fatigue
* Blurred vision
* Slow healing of cuts and wounds
* Tingling or numbness in the hands and feet
* Increased hunger
* Weight loss
* Slow movement of the eyes
* Frequent infections
* Slow healing of wounds
* Tingling or numbness in the feet and toes
* Blurred vision
* Slow healing of cuts and wounds
* Frequent infections
* Slow movement of the eyes
* Frequent infections
* Slow healing of wounds
* Tingling or numbness in the hands and feet
* Increased thirst and urination
* Blurred vision
* Slow healing of cuts and wounds
* Tingling or numbness in the hands and feet
* Increased thirst and urination
* Blurred vision
* Slow healing of cuts and wounds
* Frequent infections
* Slow movement of the eyes
* Frequent infections
* Slow healing of wounds
* Tingling or numbness in the hands